In [ ]:
!sudo apt-get install poppler-utils

In [ ]:
!pip install PyMuPDF pdf2image dotenv

In [ ]:
import json
import os
import pdf2image
from PIL import Image
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
import time
import asyncio
from dotenv import load_dotenv

# PDF parsing by using PDF2Image and GoogleGenAI
This is a useful way when there are some important informations highlighted
 
 -> No possibility to use OCR 
  
  -> Multimodal large language model as plug and play

## 1) Config variables and API Key

In [ ]:
# config global static variables

PDF_PATH = "mypdf.pdf"
OUTPUT_PNG_FOLDER = "output_folder"
CONTENT_SECTIONS = [
    {
        "section": "RADIOPROTEZIONE",
        "code": "Radioprotezione",
        "from_page": 1,
        "to_page": 10
    }
]
JSON_OUTPUT_PATH = "full_extraction.json"
JSON_METADATA_OUTPUT_PATH = "metadata_full_extraction.json"

In [ ]:
# if run locally
load_dotenv()
GOOGLE_API_KEY  = os.get_env("GOOGLE_API_KEY")

# if run on colab
# from google.colab import userdata
# GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

## 2) Extract PNG single-page image from PDF

In [ ]:
def save_pdf_as_images(pdf_path, output_folder):

  images = pdf2image.convert_from_path(pdf_path)

  if not os.path.exists(output_folder):
      os.makedirs(output_folder)

  saved_png=0

  for i,image in enumerate(images, start=1):
    output_path = f"{output_folder}/output{i}.png"
    image.save(output_path, "PNG")
    saved_png+=1
    #print(f"Image saved as PNG: {output_path}")

  print(f"Saved {saved_png} images.")
  return saved_png

In [ ]:
saved_png = save_pdf_as_images(PDF_PATH, OUTPUT_PNG_FOLDER)

## 3) Use a Vision Language Model to extract the content

In [ ]:
# config structured output class

class DomandaRisposta(BaseModel):
    num_domanda: int = Field(..., description="Numero della domanda così come riportato nell'immagine")
    domanda: str = Field(..., description="Testo completo della domanda")
    opzioni: list[str] = Field(..., description="Lista di tutte le opzioni possibili, inclusa la risposta corretta")
    risposta_corretta: str = Field(
        ..., description="Risposta evidenziata in giallo o NO_ANSWER"
    )

# config system prompt
SYSTEM_PROMPT = """
    Il seguente file contiene una serie di domande e risposte a scelta multipla. Le risposte esatte possono essere: evidenziate in giallo, oppure scritte in grassetto.
    Restituisci in output un JSON con questa struttura:
    [
        {
        "num_domanda": Numero della domanda così come riportato nell'immagine,
        "domanda": Testo della domanda completa,
        "opzioni": ["Prima opzione riportata", "Seconda opzione", eccetera]
        "risposta_corretta": La risposta evidenziata in giallo riportata fedelmente, deve essere lo stesso testo di una delle opzioni della lista
        }
    ]

    Casi possibili:
    - Caso normale (corrisponde alla maggior parte dei casi): lista di domande con risposta esatta evidenziata in giallo oppure con risposta esatta in grassetto
    - Più risposte a scelta multipla ma nessuna risposta evidenziata: risposta_corretta = "NO_ANSWER"
    - Solo la risposta corretta è presente: la risposta corretta, unica presente, è essere evidenziata in giallo. In questi casi, "opzioni" include nella lista solo la risposta corretta e "risposta_corretta" contiene lo stesso testo
    - Opzioni presenti ma nessuna domanda presente (può succedere a inizio pagina): in questo caso riempi "num_domanda" con -1 e "domanda" con "NO_QUESTION", riportando le opzioni tra "opzioni" se presente evidenziata in giallo

    Esempio concreto di input-output con quello che troverai nell'immagine:
        INPUT
        1. L'energia dei raggi x è direttamente proporzionale:
            - alla loro lunghezza d'onda
            - alla loro frequenza (evidenziata in giallo nell'immagine)
            - alla velocità della luce
            - alla loro elasticità
        2. L'imaging plate è:
            - un detettore che registra sulla propria superficie l'energia dei fotoni x (evidenziato in giallo nell'immagine)
            - un tipo di tubo radiogeno
            - un algoritmo di ricostruzione delle immagini in tac
            - un sistema integrato di appiattimento delle immagini
        3. I raggi X persistono nella sala radiologica, una volta interrotta la esposizione radiante, per:
            Scompaiono immediatamente (evidenziata in giallo)
        4. Per cosa non viene utilizzata la via intradermica?
            - Prova di reazione alla tubercolina
            - Test cutanei
            - Desensibilizzazione
            - Terapia immunosoppressiva

        OUTPUT
        [
            {
            "num_domanda": -1,
            "domanda": "NO_QUESTION",
            "opzioni": [
                "verificare l'identità dell'assistito in modo attivo attraverso il braccialetto identificativo e il braccialetto con codice colore della trasfusione",
                "verificare l'identità dell'assistito in modo attivo, controllare che il numero della cartella clinica corrisponda a quello del braccialetto identificativo",
                "verificare l'identità dell'assistito in modo attivo e controllare che in cartella clinica sia presente la richiesta di trasfusione"
            ],
            "risposta_corretta": "NO_ANSWER"
            },
            {
            "num_domanda": 1,
            "domanda": "L'energia dei raggi x è direttamente proporzionale:",
            "opzioni": [
                "alla loro lunghezza d'onda",
                "alla loro frequenza",
                "alla velocità della luce",
                "alla loro elasticità"
            ],
            "risposta_corretta": "alla loro frequenza"
            },
            {
            "num_domanda": 2,
            "domanda": "L'imaging plate è:",
            "opzioni": [
                "un detettore che registra sulla propria superficie l'energia dei fotoni x",
                "un tipo di tubo radiogeno",
                "un algoritmo di ricostruzione delle immagini in tac",
                "un sistema integrato di appiattimento delle immagini"
            ],
            "risposta_corretta": "un detettore che registra sulla propria superficie l'energia dei fotoni x"
            },
            {
            "num_domanda": 3,
            "domanda": "I raggi X persistono nella sala radiologica, una volta interrotta la esposizione radiante, per:",
            "opzioni": [
                "Scompaiono immediatamente"
            ],
            "risposta_corretta": "Scompaiono immediatamente"
            },
            {
            "num_domanda": 4,
            "domanda": "Per cosa non viene utilizzata la via intradermica?",
            "opzioni": [
                "Prova di reazione alla tubercolina",
                "Test cutanei",
                "Desensibilizzazione",
                "Terapia immunosoppressiva"
            ],
            "risposta_corretta": "NO_ANSWER"
            }
        ]
  """

### 3.1) Use Gemini (plug-and-play)

In [ ]:
# config client
CLIENT = genai.Client(api_key=GOOGLE_API_KEY)

# config models (list with decreasing quality)

GEMINI_MODELS = [
    "gemini-3-flash-preview",
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.0-flash",
]

MIL_TOKEN=1000000
PRICING = {
    "gemini-3-flash-preview":{
        "input":0.5/MIL_TOKEN,
        "output":3/MIL_TOKEN
    },
    "gemini-2.5-flash":{
        "input":0.3/MIL_TOKEN,
        "output":2.5/MIL_TOKEN
    },
    "gemini-2.5-flash-lite":{
        "input":0.1/MIL_TOKEN,
        "output":0.4/MIL_TOKEN
    },
    "gemini-2.0-flash":{
        "input":0.1/MIL_TOKEN,
        "output":0.4/MIL_TOKEN
    }
}

In [ ]:
def is_resource_exhausted(error: Exception) -> bool:
    error_str = str(error).lower()
    return any(k in error_str for k in [
        "resourceexhausted",
        "quota exceeded",
        "rate limit",
        "429"
    ])

def generate_with_fallback(uploaded_file):
    last_exception = None

    for model_name in GEMINI_MODELS:
        try:
          start = time.time()
          response = CLIENT.models.generate_content(
              model=model_name,
              contents=[uploaded_file],
              config={
                  "response_mime_type": "application/json",
                  "response_schema": list[DomandaRisposta],
                  "temperature": 0,
                  "system_instruction": SYSTEM_PROMPT,
                  "thinking_config": {
                      "include_thoughts": False
                  }
              }
          )
          end = time.time()

          try:
            input_cost = response.usage_metadata.prompt_token_count*PRICING[model_name]["input"]
            if response.usage_metadata.thoughts_token_count:
              output_cost = (response.usage_metadata.candidates_token_count + response.usage_metadata.thoughts_token_count)*PRICING[model_name]["output"]
            else:
              output_cost = (response.usage_metadata.candidates_token_count)*PRICING[model_name]["output"]

            response_metadata = {
                "time": end-start,
                "input_cost": input_cost,
                "output_cost": output_cost
            }
          except Exception as e:
            print(f"Error getting metadata: {e}")
            response_metadata = {
                "time": end-start,
                "input_cost": 0,
                "output_cost": 0
            }
          finally:
            # sleep in caso di successo
            time.sleep(15)
            return response, response_metadata

        except Exception as e:
            last_exception = e

            if not is_resource_exhausted(e):
                raise e  # errore non gestibile → fail immediato

            print(f"Resource exhausted with {model_name}, trying next model...")
            # time.sleep(15)

    # se arriviamo qui, tutti i modelli hanno fallito
    raise RuntimeError("All Gemini models exhausted") from last_exception

In [ ]:
answers = []
response_metadata_list = []
start_index = 1

for i in range(start_index, saved_png + 1):
  print("Working with image:", i)

  image_path = f"{OUTPUT_PNG_FOLDER}/output{i}.png"

  # upload immagine a Gemini
  uploaded_file = CLIENT.files.upload(file=image_path)

  response, response_metadata = generate_with_fallback(uploaded_file)
  raw_text = response.text

  print("Answer generated for image", i)

  # =========================
  # PARSE JSON
  # =========================
  try:
    # ANSWERS
    parsed = json.loads(raw_text)
    page_json = {
        "num_page": i,
        "items": parsed
    }
    answers.append(page_json)
  except Exception as e:
    print(f"JSON parsing error on page {i}: {e}")

  try:
    # METADATA
    response_metadata["page"]=i
    response_metadata_list.append(response_metadata)
  except Exception as e:
    print(f"JSON metadata parsing error on page {i}: {e}")

  # =========================
  # SAVE JSON (incrementale)
  # =========================
  with open(JSON_OUTPUT_PATH, "w") as f:
      json.dump(answers, f, indent=2, ensure_ascii=False)

  with open(JSON_METADATA_OUTPUT_PATH, "w") as f:
      json.dump(response_metadata_list, f, indent=2, ensure_ascii=False)

  print(f"Processing completed. JSON saved in {JSON_OUTPUT_PATH}, METADATA JSON saved in {JSON_METADATA_OUTPUT_PATH}")

### 3.2) [ALTERNATIVE] Use QWEN-2.5-VL-7B (from Unsloth)
Uncomment the code if you want to use QWEN as local model (requires GPU L4 or more powerful) 

In [ ]:
# %%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     !pip install unsloth
# else:
#     # Do this only in Colab notebooks! Otherwise use pip install unsloth
#     import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
#     xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
#     !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
#     !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     !pip install --no-deps unsloth
# !pip install transformers==4.56.2
# !pip install --no-deps trl==0.22.2

In [ ]:
# from unsloth import FastVisionModel
# import torch

# # Carica il modello pre-addestrato (già in 4-bit per efficienza)
# # Questo è lo stesso modello 'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit' usato nel notebook
# model, tokenizer = FastVisionModel.from_pretrained(
#     "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit",
#     load_in_4bit = True,
# )

# FastVisionModel.for_inference(model) # Abilita la modalità di inferenza

In [ ]:
# def generate_with_qwen():
#     answers=[]
#     for i in range(1,saved_png+1):
#         print("Working with image: ", i)

#         image = Image.open(f"{OUTPUT_PNG_FOLDER}/output{i}.png").convert('RGB')

#         messages = [
#             {"role": "user", "content": [
#                 {"type": "image"},
#                 {"type": "text", "text": SYSTEM_PROMPT}
#             ]}
#         ]

#         input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
#         inputs = tokenizer(
#             image,
#             input_text,
#             add_special_tokens = False,
#             return_tensors = "pt",
#         ).to("cuda")

#         from transformers import TextStreamer
#         text_streamer = TextStreamer(tokenizer, skip_prompt = True)
#         output_ids = model.generate(**inputs, use_cache = True)

#         answer = tokenizer.decode(
#             output_ids[0][inputs["input_ids"].shape[-1]:],
#             skip_special_tokens=True
#         )

#         print("Answer generated for image", i)        
        
#         try:
#             page_json = {"num_page":i, "items":json.loads(answer)}
#             answers.append(page_json)
#         except Exception as e:
#             print(f"Error parsing json in page {i}, trying to recover: {e}")
#             #try stripping ```json at start and ``` at end
#             try:
#                 answer = answer.lstrip("```json").rstrip("```")
#                 page_json = {"num_page":i, "items":json.loads(answer)}
#                 answers.append(page_json)
#             except Exception as e:
#                 print(f"Error parsing json in page {i}, giving up: {e}")

#         # save json
#         with open(JSON_OUTPUT_PATH, 'w') as f:
#             json.dump(answers, f, indent=4)

In [ ]:
# generate_with_qwen()

## 4) Data augmentation of the questions using GEMMA-3-27B-IT

In [ ]:
# Limite di 25 chiamate al minuto
sem = asyncio.Semaphore(25)

async def opzioni_augmentation_async(domanda: str, risposta_corretta: str) -> list[str]:
    async with sem:

        # Struttura Few-Shot
        messages = [
            types.Content(role="user", parts=[types.Part(text="Sei un assistente esperto in creazione di test a scelta multipla. Data la domanda e la risposta esatta nel formato DOMANDA | RISPOSTA ESATTA, Genera TRE opzioni errate plausibili ma chiaramente incorrette.")]),
            types.Content(role="user", parts=[types.Part(text="Domanda: 'Molecola bersaglio:' | Risposta corretta: 'DNA'")]),
            types.Content(role="model", parts=[types.Part(text="RNA <SEP> H2O <SEP> CO2")]),
            types.Content(role="user", parts=[types.Part(text="Domanda: 'L’effetto deterministico:' | Risposta corretta: 'È soglia dipendente'")]),
            types.Content(role="model", parts=[types.Part(text="È soglia indipendente <SEP> È soglia randomica <SEP> È soglia deterministica")]),
            # Input attuale
            types.Content(role="user", parts=[types.Part(text=f"Domanda: '{domanda}' | Risposta corretta: '{risposta_corretta}'")])
        ]

        try:
            # Nota: assicurati che l'SDK supporti l'invocazione async (solitamente .aio)
            response = await CLIENT.aio.models.generate_content(
                model="gemma-3-27b-it",
                contents=messages
            )

            text = response.text.strip()
            opzioni_errate = [opt.strip() for opt in text.split("<SEP>")]
            return opzioni_errate
        except Exception as e:
            print(f"Errore durante la chiamata API: {e}")
            return []

async def process_modules(full_file_json):
      for d in full_file_json:
          domanda = d["domanda"]
          risposta_corretta = d["risposta_corretta"]
          opzioni = d["opzioni"]

          if len(opzioni) == 1 and risposta_corretta == opzioni[0]:
              print(f"Arricchisco domanda {d['num_domanda']}...")

              nuove_opzioni = await opzioni_augmentation_async(domanda, risposta_corretta)

              if len(nuove_opzioni) == 3:
                  opzioni.extend(nuove_opzioni)
                  # Salvataggio incrementale ad ogni successo
                  with open(f"{JSON_OUTPUT_PATH}_new.json", "w", encoding="utf-8") as f:
                      json.dump(d, f, indent=2, ensure_ascii=False)
              else:
                  print(f"Errore formato per domanda {d['num_domanda']}")

              # Delay opzionale per gestire meglio il rate limit se necessario
              await asyncio.sleep(3)

In [ ]:
with open(JSON_OUTPUT_PATH, "r", encoding="utf-8") as f:
    full_file_json = json.load(JSON_OUTPUT_PATH)

await process_modules(full_file_json)

## 5) Separate the json file in different json files based on the section

In [ ]:
# Carica il JSON completo
with open(JSON_OUTPUT_PATH, "r", encoding="utf-8") as f:
    full_json = json.load(f)

# Per ogni sezione definita in CONTENT_SECTIONS
for section_info in CONTENT_SECTIONS:
    section_name = section_info["section"]
    section_code = section_info["code"]
    from_page = section_info["from_page"]
    to_page = section_info["to_page"]
    
    print(f"Processando sezione: {section_name} (pagine {from_page}-{to_page})")
    
    # Filtra le pagine che appartengono a questa sezione
    section_data = []
    for page_data in full_json:
        page_num = page_data["num_page"]
        if from_page <= page_num <= to_page:
            section_data.append(page_data)
    
    # Salva il JSON della sezione
    output_filename = f"{section_name.lower()}_final.json"
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(section_data, f, indent=2, ensure_ascii=False)
    
    print(f"Salvato {output_filename} con {len(section_data)} pagine")

print("\nDivisione completata!")